In [0]:
-- Find batsman with highest percentage of team runs in a match
with batsman_run as (
  select sum(batsman_runs) as runs, match_id, striker
  from deliveries
  group by match_id,striker
),
team_run as (
select sum(total_runs) as teams_run,match_id
from deliveries 
group by match_id
),
result as (
  select t.match_id,b.striker,round(b.runs*100/t.teams_run,2) as per,
 rank()over(partition by b.match_id order by (b.runs*100/t.teams_run) desc) as rank
from batsman_run b
join team_run t
on b.match_id = t.match_id
)
select * from result where rank = 1;


-- Find matches where a single batsman scored more than 50% of team total. 
with batsman_run as (
  select sum(batsman_runs) as runs, match_id, striker
  from deliveries
  group by match_id,striker
),
team_run as (
select sum(total_runs) as teams_run,match_id
from deliveries 
group by match_id
),
result as (
  select b.match_id, round(b.runs*100/t.teams_run,2) as per
from batsman_run b
join team_run t
on b.match_id = t.match_id
)
select distinct match_id from result where per > 50;

-- Calculate average partnership runs per team per season. 
with wicket_fall as (
  select *,
  case when is_wicket = true then 1 else 0 end as wicket
  from deliveries
),
partnership_id as (
  select *,sum(wicket) over(partition by match_id,innings order by over,ball) as partnership_id
  from wicket_fall
),
partnership as (
  select match_id, innings, batting_team, partnership_id,
    sum(total_runs) as runs
  from partnership_id
  group by match_id, innings, batting_team, partnership_id
)
select season, batting_team, round(avg(runs),2) as avg_runs
from partnership p
join matches m
on p.match_id = m.match_id
group by season, batting_team
order by season, batting_team;

-- Find bowlers who never conceded a six in a season
with bowler_sixes as (
  select m.season, d.bowler,
    sum(case when d.batsman_runs = 6 then 1 else 0 end) as sixes_conceded
  from data.ipldata.deliveries d
  join data.ipldata.matches m on d.match_id = m.match_id
  group by m.season, d.bowler
)
select season, bowler
from bowler_sixes
where sixes_conceded = 0;

-- Find batsmen who faced more than 1000 balls but never scored a century
with batsman_balls as (
  select striker as batsman, count(*) as balls_faced
  from data.ipldata.deliveries
  group by striker
),
centuries as (
  select striker as batsman, match_id, sum(batsman_runs) as runs
  from data.ipldata.deliveries
  group by striker, match_id
  having sum(batsman_runs) >= 100
)
select b.batsman
from batsman_balls b
left join centuries c on b.batsman = c.batsman
where b.balls_faced > 1000 and c.batsman is null;

-- Find team with highest average powerplay score (overs 1–6)
with powerplay_runs as (
  select batting_team, match_id, innings, sum(total_runs) as powerplay_score
  from data.ipldata.deliveries
  where over between 1 and 6
  group by batting_team, match_id, innings
)
select batting_team, round(avg(powerplay_score),2) as avg_powerplay_score
from powerplay_runs
group by batting_team
order by avg_powerplay_score desc
limit 1;

-- Find matches where both teams scored more than 200 runs
with team_scores as (
  select match_id, innings, batting_team, sum(total_runs) as team_score
  from data.ipldata.deliveries
  group by match_id, innings, batting_team
)
select match_id
from team_scores
group by match_id
having count(*) = 2 and min(team_score) > 200;

-- Find teams that successfully defended totals under 150
select m.season, m.match_id, m.team1 as defending_team, m.first_innings_score
from data.ipldata.matches m
where m.first_innings_score < 150 and m.winner = m.team1 and m.win_by = 'Runs';






